# 03. Model Training & Evaluation Benchmark

## Mục tiêu
Thực hiện huấn luyện và đánh giá 25 tổ hợp mô hình:
- **5 Mô hình**: Logistic Regression, Random Forest, XGBoost, CatBoost, ANN (PyTorch).
- **5 Kỹ thuật Imbalance**: SMOTE, SMOTE-ENN, ADASYN, Borderline-SMOTE, Class Weighting.

### Các ràng buộc quan trọng (spec mục 5 & 8):
1. **CatBoost dùng chung features đã encode sẵn**: KHÔNG truyền `cat_features` để đảm bảo so sánh 1:1 công bằng với các mô hình khác.
2. **Tập test KHÔNG resample**: Đánh giá trên phân phối thực tế.
3. **PR-AUC là metric chính**: Không dùng ROC-AUC làm chỉ số quyết định.

In [ ]:
# 1. Setup & Imports
import sys
from pathlib import Path
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.config import (
    PROCESSED_DATA_DIR, RESULTS_DIR, SEED, TARGET_COL,
    MODEL_NAMES, IMBALANCE_TECHNIQUES
)
from src.imbalance.resamplers import apply_imbalance
from src.models.train import build_model, train_model, predict_proba
from src.evaluation.metrics import evaluate_model

print('Models:', MODEL_NAMES)
print('Imbalance techniques:', IMBALANCE_TECHNIQUES)

In [ ]:
# 2. Load Processed Data
train_path = PROCESSED_DATA_DIR / 'train_encoded.parquet'
test_path = PROCESSED_DATA_DIR / 'test_encoded.parquet'

print('Đang load dữ liệu tiền xử lý...')
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

# Tách X, y
X_train_raw = train_df.drop(columns=[TARGET_COL]).values
y_train_raw = train_df[TARGET_COL].values
X_test = test_df.drop(columns=[TARGET_COL]).values
y_test = test_df[TARGET_COL].values

print(f'Train features: {X_train_raw.shape}, Test features: {X_test.shape}')
print(f'Test fraud rate: {y_test.mean():.4%}')

--- 
## 3. Chạy Benchmark 25 Tổ Hợp
Lặp qua 5 models × 5 kỹ thuật imbalance.

In [ ]:
# 3. Benchmark Loop
benchmark_results = []

for model_name in MODEL_NAMES:
    for technique in IMBALANCE_TECHNIQUES:
        combo_name = f'{model_name} + {technique}'
        print(f'--> Training: {combo_name}...')
        
        try:
            # 1. Apply Imbalance
            X_res, y_res, class_weights = apply_imbalance(
                technique, X_train_raw, y_train_raw, seed=SEED
            )
            
            # 2. Build & Train Model
            model = build_model(
                model_name=model_name,
                input_dim=X_res.shape[1],
                class_weights=class_weights,
                seed=SEED,
            )
            trained_model = train_model(model, X_res, y_res, model_name=model_name)
            
            # 3. Predict & Evaluate trên Test Set
            y_proba = predict_proba(trained_model, X_test, model_name=model_name)
            metrics = evaluate_model(y_test, y_proba)
            
            row = {
                'model': model_name,
                'imbalance_technique': technique,
                **metrics
            }
            benchmark_results.append(row)
            print(f'    PR-AUC: {metrics["pr_auc"]:.4f} | F1: {metrics["f1"]:.4f} | F2: {metrics["f2"]:.4f}')
        except Exception as e:
            print(f'    Lỗi khi chạy {combo_name}: {e}')

df_results = pd.DataFrame(benchmark_results)
print('Benchmark hoàn tất!')

--- 
## 4. Bảng Kết Quả & So Sánh Hiệu Năng
Tổng hợp kết quả PR-AUC, F1, F2 và ROC-AUC.

In [ ]:
# 4. Hiển thị bảng so sánh
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df_results.to_csv(RESULTS_DIR / 'model_benchmark_results.csv', index=False)

# Pivot table PR-AUC theo Model và Kỹ thuật Imbalance
pivot_prauc = df_results.pivot(index='model', columns='imbalance_technique', values='pr_auc')
print('=== BẢNG PR-AUC BENCHMARK (METRIC CHÍNH) ===')
display(pivot_prauc.style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# 5. Trực quan hóa PR-AUC
plt.figure(figsize=(12, 6))
sns.barplot(data=df_results, x='model', y='pr_auc', hue='imbalance_technique')
plt.title('So Sánh PR-AUC Giữa Các Mô Hình & Kỹ Thuật Imbalance', fontsize=14, fontweight='bold')
plt.ylabel('PR-AUC (Metric Chính)')
plt.xlabel('Mô Hình')
plt.ylim(0, 1.0)
plt.legend(title='Kỹ Thuật Imbalance', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'benchmark_prauc_comparison.png', dpi=300)
plt.show()

--- 
## 5. Kết Luận Bước 3
1. Xác định mô hình có PR-AUC cao nhất (thường là XGBoost hoặc CatBoost).
2. Đánh giá kỹ thuật imbalance nào cho hiệu năng phân loại cao nhất.
3. **Câu hỏi nghiên cứu CIES**: Liệu kỹ thuật cho PR-AUC cao nhất có giữ được độ tin cậy giải thích (CIES) cao nhất không, hay phải đánh đổi?